In [6]:
# -*- coding: utf-8 -*-
"""
movie_long_micro_panel.csv  →  staggered_did_weekly_min.dta
- CSV 등 부가 저장 없음
- 윈도우(리드/랙) 적용하지 않음
- 사전=실관람, 사후=OTT 규칙으로 필터
- 주단위(y=평균 별점)로 집계, csdid 최소 스키마만 저장

Stata 예:
  use "staggered_did_weekly_min.dta", clear
  csdid y, ivar(movie_id) time(weeknum) gvar(G) wboot reps(999) seed(20250819)
"""

import pandas as pd
import numpy as np
from pathlib import Path

INPUT = Path("movie_long_micro_panel.csv")
OUT_DTA = Path("staggered_did_weekly_min.dta")

# 1) CSV 로드 (utf-8 → cp949 순서로 시도)
df = None; last_err = None
for enc in ("utf-8", "cp949"):
    try:
        df = pd.read_csv(INPUT, encoding=enc, low_memory=False)
        print(f"[info] read ok: {enc}, rows={len(df):,}")
        break
    except Exception as e:
        last_err = e
if df is None:
    raise RuntimeError(f"read fail: {last_err}")

cols = df.columns.tolist()

# 2) 컬럼 매핑 (여유 있는 후보명)
def pick(cands, cols):
    for c in cands:
        if c in cols: return c
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None

movie = pick(["영화명","영화","title","movie","movie_title"], cols)
date  = pick(["일자","작성일","리뷰작성일","date","review_date"], cols)
post  = pick(["post_netflix","post_netlfix","post"], cols)
ott   = pick(["OTT관람객","ott_viewer","is_ott_viewer","ott_flag"], cols)
thea  = pick(["실관람객","theater_viewer","is_theater_viewer","theater_flag"], cols)
rate  = pick(["별점","rating","stars","score"], cols)

need = [movie, date, post, ott, thea, rate]
assert all(c is not None for c in need), \
    f"missing cols: {dict(movie=movie,date=date,post=post,ott=ott,thea=thea,rate=rate)}"

use = df[[movie, date, post, ott, thea, rate]].copy()
use.columns = ["movie_name","date","post","ott_viewer","theater_viewer","rating"]

# 3) 타입 정리
use["date"] = pd.to_datetime(use["date"], errors="coerce")
for c in ["post","ott_viewer","theater_viewer","rating"]:
    use[c] = pd.to_numeric(use[c], errors="coerce")

# 4) 필터: 사전=실관람만, 사후=OTT만
sel = ((use["post"]==1) & (use["ott_viewer"]==1)) | ((use["post"]==0) & (use["theater_viewer"]==1))
use = use.loc[sel].dropna(subset=["date"]).copy()

# 5) 주 인덱스 (월요일 앵커 → Stata weeknum: 1960-01-04(월) 기준)
use["week_monday"] = use["date"] - pd.to_timedelta(use["date"].dt.weekday, unit="D")
base_mon = pd.Timestamp("1960-01-04")
use["weeknum"] = ((use["week_monday"] - base_mon).dt.days // 7).astype(int)

# 6) 영화 numeric id (문자열은 DTA에 안 넣음: 인코딩 이슈 회피)
codes, uniq = pd.factorize(use["movie_name"], sort=True)
use["movie_id"] = codes + 1

# 7) 코호트 G = 영화별 최초 처치 주(최초 post==1 주)
g = use.loc[use["post"]==1].groupby("movie_id")["weeknum"].min().rename("G")
use = use.merge(g, on="movie_id", how="left")     # never-treated는 G=NaN 유지 (대조군으로 활용)

# 8) 주단위 집계: y=평균 별점, n_reviews=리뷰 수
weekly = (use.groupby(["movie_id","weeknum","G"], as_index=False)
          .agg(y=("rating","mean"),
               n_reviews=("rating","count")))

# 9) 저장: csdid 최소 스키마만 (문자열 없음 → 용량 작고 Stata 호환 안정)
weekly[["movie_id","weeknum","G","y","n_reviews"]].to_stata(
    OUT_DTA.as_posix(), write_index=False, version=118
)

print("[done]", OUT_DTA, "rows=", len(weekly))
print("movies (all):", weekly["movie_id"].nunique())
print("treated movies (have G):", weekly.loc[weekly["G"].notna(), "movie_id"].nunique())


[info] read ok: utf-8, rows=437,133
[done] staggered_did_weekly_min.dta rows= 5787
movies (all): 168
treated movies (have G): 168
